# Coralscapes zero-shot evaluation on Moreton Bay annotations

Scores a pretrained Coralscapes SegFormer, with no fine-tuning, against the expert CVAT annotations of the Myora Reef BlueROV2 frames. Cells are meant to be run from top to bottom. Everything prints or plots as it runs, so each frame can be watched as it is scored.

| Section | What it does |
|---|---|
| Parameters | Paths and run options |
| Setup | Device, config and helper functions |
| Load the model | Loads the checkpoint and shows how its classes map onto the project classes |
| Convert the CVAT export | Turns the CVAT export into class-index masks and previews the annotations |
| Zero-shot evaluation | Frame-by-frame scoring, per-class metrics and confusion matrix |
| Raw model classes | What the model predicted on the annotated pixels, and a coarse coral versus not-coral score |

Runs on an NVIDIA GPU, Apple GPU (MPS) or CPU. Inference follows the official Coralscapes approach (see `coralscapes/`): `SegformerImageProcessor` and plain `from_pretrained`, short side resized to 1024, a 1024x1024 sliding window with logit averaging, fp32. The shared code lives in `utils/engine.py` and `utils/labels.py`.

The next notebook in the workflow is `coral_enhancement_comparison.ipynb`, which reuses the masks written here.

## Parameters

In [ ]:
PROJECT_DIR = "."                         # the zeroshot folder (where this notebook lives)
WORK_DIR    = "./work"                    # outputs go here (git-ignored)
CVAT_EXPORT = "../data/cvat_annotations"  # UNZIPPED CVAT "Segmentation mask 1.1" export
CONFIG_PATH = "configs/config.yaml"

MAX_PREVIEW = 12                          # maximum number of frames to draw in preview grids

## Setup

In [ ]:
import os, sys, json, time
from pathlib import Path

PROJECT_DIR = Path(PROJECT_DIR).expanduser().resolve()
os.chdir(PROJECT_DIR)                      # config paths such as configs/config.yaml are relative to this folder
WORK_DIR = Path(WORK_DIR).expanduser().resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
sys.path.append("../utils")                # shared modules live in the repository's utils folder

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from PIL import Image
from IPython.display import display

from engine import load_reference, metrics_from_confusion, pick_device, segment_image, update_confusion
from labels import load_config, pick_mapping, resolve_id2label
from prepare_cvat_export import prepare

plt.rcParams["figure.dpi"] = 100
device = pick_device()
cfg = load_config(CONFIG_PATH)
names, colours, IGN = cfg["classes"]["names"], cfg["classes"]["colours"], cfg["classes"]["ignore_index"]
K = len(names)
PAL = np.vstack([colours, [[90, 90, 90]]]).astype(np.uint8)     # last colour = "model predicted a class outside the project scheme"

print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda"
      else "Apple GPU (MPS)" if device.type == "mps" else "CPU (fine for a few frames)")
print("work dir:", WORK_DIR)
print("model:", cfg["model"]["checkpoint"])
print("project classes:", names)



def blend(img, m, alpha=0.55, pal=PAL):
    out = img.copy()
    ok = (m != IGN) & (m < len(pal))
    out[ok] = (alpha * pal[m[ok]] + (1 - alpha) * img[ok]).astype(np.uint8)
    return out


def legend(ax=None, ncol=5):
    h = [Patch(color=colours[i] / 255, label=n) for i, n in enumerate(names)]
    h.append(Patch(color=PAL[K] / 255, label="unmapped (model predicted a class outside the project scheme)"))
    (ax or plt.gcf()).legend(handles=h, loc="lower center", ncol=ncol, frameon=False, fontsize=8)

## Load the model and check how its classes map onto the project classes

Loads the checkpoint named in `configs/config.yaml` (`model.checkpoint`; the first run downloads it). The model's class names come from the checkpoint itself, and the `v1_coralscapes_mapping` section of the config (matched by **name**) turns them into project classes. If a name in the mapping does not exist in the model, the cell stops and prints the full class list.

The table below should be checked: the grouping is a deliberate choice. Model classes that are not listed are ignored, never scored, and counted as "unmapped" when the model predicts them.

In [ ]:
pd.set_option("display.max_colwidth", None)   # show full class lists in the table
t0 = time.time()
# SegformerImageProcessor + SegformerForSemanticSegmentation.from_pretrained, as in the official Coralscapes approach
preprocessor, zs_model = load_reference(cfg["model"]["checkpoint"], device,
                                        id2label_file=cfg["model"].get("id2label_file"))
print("preprocessor size:", preprocessor.size, "| mean:", preprocessor.image_mean, "| std:", preprocessor.image_std)
print(f"loaded {cfg['model']['checkpoint']} in {time.time() - t0:.0f}s, "
      f"{sum(p.numel() for p in zs_model.parameters()) / 1e6:.0f}M parameters")

id2label, src = resolve_id2label(zs_model, cfg)
map_key, lut = pick_mapping(id2label, cfg, names)          # model class id -> project class (255 = ignored)
print(f"class names from: {src} | model outputs: {zs_model.config.num_labels} | using mapping: '{map_key}'\n")

rows = [{"id": i, "model class": n, "→ project class": "(ignored)" if lut[i] == 255 else names[lut[i]]}
        for i, n in sorted(id2label.items())]
df = pd.DataFrame(rows)
mapped = df[df["→ project class"] != "(ignored)"].groupby("→ project class")["model class"].apply(lambda s: ", ".join(s))
display(mapped.reindex(names).fillna("—").to_frame())
ignored = df[df["→ project class"] == "(ignored)"]["model class"].tolist()
print(f"{len(ignored)} model classes are not in the mapping (ignored):")
print(", ".join(ignored))

## Convert the CVAT export and preview the annotations

Reads `SegmentationClass/` and `labelmap.txt` from the export, turns colours into class indices (unannotated pixels become the ignore index), and prints how each CVAT label was mapped. The printout should be checked against the labels used in CVAT.

In [ ]:
export_dir = Path(CVAT_EXPORT).expanduser()
if not export_dir.exists():
    raise FileNotFoundError(f"CVAT_EXPORT not found: {export_dir}. Unzip the CVAT export into data/cvat_annotations or point CVAT_EXPORT at it.")
own = Path(cfg["data"]["own_dir"])
prepare(CONFIG_PATH, export_dir, own)

In [ ]:
kept = json.load(open(own / "summary.json"))["kept"]


def find_image(stem):
    return next(p for p in (own / "images").glob(f"{stem}.*"))


n = min(len(kept), MAX_PREVIEW)
cols = 3
fig, axes = plt.subplots((n + cols - 1) // cols, cols, figsize=(15, 4.2 * ((n + cols - 1) // cols)), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
for ax, s in zip(axes.ravel(), kept[:n]):
    img = np.array(Image.open(find_image(s)).convert("RGB"))
    ax.imshow(blend(img, np.array(Image.open(own / "masks" / f"{s}.png"))))
    ax.set_title(s, fontsize=8)
legend(ncol=5)
plt.suptitle(f"CVAT annotations ({len(kept)} frames; unlabelled areas shown untouched)", y=0.98)
plt.tight_layout(rect=(0, 0.05, 1, 0.97))
plt.show()

## Zero-shot evaluation

The pretrained Coralscapes model, **without fine-tuning**, remapped to the project classes and scored on every annotated frame. Scoring is at the frame's original resolution, on annotated pixels only.

How to read the results:
- **IoU, recall and precision** are per class. Recall answers "of the pixels annotated as this class, how many did the model find". With no background annotated, **precision is optimistic**, because wrong predictions on unannotated pixels are never counted.
- **Unmapped** (grey) means the model predicted a class outside the project scheme (human, transect line, millepora and so on). It counts as wrong on annotated pixels.
- With only a handful of frames, the overlays are more trustworthy than the numbers.

In [ ]:
ZS_LUT = np.where(lut == 255, K, lut).astype(np.int64)         # model ids -> project classes; unused -> K ("unmapped")


RAW_IDS = {}   # stem -> the model's own class ids from the zero-shot run (used by the raw-class analysis below)


def predict_frame(model, img, id_lut, key):
    """Sliding-window inference (short side 1024, 1024x1024 windows, averaged logits, fp32)."""
    raw = segment_image(Image.fromarray(img), preprocessor, model, device)
    RAW_IDS[key] = raw
    return id_lut[raw.astype(np.int64)]


def evaluate_model(model, stems, id_lut, title="", show=True):
    conf = np.zeros((K, K + 1), np.int64)
    rows = []
    for i, s in enumerate(stems, 1):
        t = time.time()
        img = np.array(Image.open(find_image(s)).convert("RGB"))
        gt = np.array(Image.open(own / "masks" / f"{s}.png"))
        pred = predict_frame(model, img, id_lut, key=s)
        fc = update_confusion(np.zeros((K, K + 1), np.int64), gt, pred, K, IGN)
        conf += fc
        acc = np.trace(fc[:, :K]) / max(fc.sum(), 1)
        rows.append({"frame": s, "labelled_px_accuracy": acc})
        print(f"[{i}/{len(stems)}] {s}: labelled-pixel accuracy {acc:.1%}  ({time.time() - t:.1f}s)", flush=True)
        if show:
            fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
            for a, im, tt in zip(ax, [img, blend(img, gt), blend(img, pred)], ["Frame", "CVAT Labels", f"Prediction {title}"]):
                a.imshow(im); a.set_title(tt, fontsize=9); a.axis("off")
            legend(ncol=5)
            plt.tight_layout(rect=(0, 0.07, 1, 1))
            plt.show()
    return conf, metrics_from_confusion(conf, names), pd.DataFrame(rows)


zs_conf, zs_metrics, zs_frames = evaluate_model(zs_model, kept, ZS_LUT, "(zero-shot)")

In [ ]:
def metrics_table(m):
    return pd.DataFrame(m["per_class"]).T.assign(
        iou=lambda d: d["iou"].astype(float), recall=lambda d: d["recall"].astype(float),
        precision=lambda d: d["precision"].astype(float))[["gt_pixels", "iou", "recall", "precision"]]

EXCLUDE = ["sand", "rubble", "background"]          # classes whose labelled pixels are left out of scoring

zs_conf_ex = zs_conf.copy()
zs_conf_ex[[names.index(n) for n in EXCLUDE], :] = 0          # drop those classes' ground-truth pixels
zs_metrics_ex = metrics_from_confusion(zs_conf_ex, names)     # mIoU / accuracy now ignore them too

tbl = metrics_table(zs_metrics_ex).drop(index=EXCLUDE)
print(f"mIoU (classes present, excluding {EXCLUDE}): {zs_metrics_ex['mIoU']:.3f}   "
      f"labelled-pixel accuracy: {zs_metrics_ex['pixel_acc']:.3f}")
display(tbl.style.format({"iou": "{:.3f}", "recall": "{:.3f}", "precision": "{:.3f}", "gt_pixels": "{:,}"}, na_rep="n/a")
        .background_gradient(subset=["iou", "recall", "precision"], cmap="RdYlGn", vmin=0, vmax=1))

fig, ax = plt.subplots(figsize=(9, 3.2))
tbl[["iou", "recall", "precision"]].plot.bar(ax=ax, ylim=(0, 1), width=0.8)
ax.set_title("Zero-shot, per class (coral and marine species only)"); ax.set_xlabel(""); ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

In [ ]:
cols = names + ["unmapped"]
rec = zs_conf / np.maximum(zs_conf.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(8, 5.5))
im = ax.imshow(rec, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(K + 1), cols, rotation=40, ha="right"); ax.set_yticks(range(K), names)
for i in range(K):
    for j in range(K + 1):
        if zs_conf[i].sum() and rec[i, j] >= 0.02:
            ax.text(j, i, f"{rec[i, j]:.0%}", ha="center", va="center", fontsize=8,
                    color="white" if rec[i, j] > 0.5 else "black")
ax.set_xlabel("model predicted"); ax.set_ylabel("annotated class")
ax.set_title("What the model called each annotated class (row-normalised)")
plt.colorbar(im, fraction=0.04); plt.tight_layout(); plt.show()

print("Most common confusion per class:")
for i, n in enumerate(names):
    if zs_conf[i].sum() == 0:
        continue
    off = rec[i].copy(); off[i] = 0
    j = int(off.argmax())
    print(f"  {n:15s} correct {rec[i, i]:>4.0%} | most often called '{cols[j]}' ({rec[i, j]:.0%})")

In [ ]:
out_dir = WORK_DIR / "eval_notebook"
out_dir.mkdir(parents=True, exist_ok=True)
tbl.to_csv(out_dir / "zeroshot_per_class.csv")
zs_frames.to_csv(out_dir / "zeroshot_per_frame.csv", index=False)
np.savetxt(out_dir / "zeroshot_confusion.csv", zs_conf, fmt="%d", delimiter=",", header=",".join(cols))
print("saved to", out_dir)

## Raw model classes on the annotated pixels

The table above scores the model *after* translating its classes into the project classes. This section shows the translation-free view: for every annotated region, which of the model's own classes it actually predicted, pooled over all frames. It also gives a coarse **coral versus not-coral** score, which is the distinction a V1 model can realistically make (it has no encrusting class and only a catch-all `other coral *`).

Only annotated pixels are counted, so the "not-coral" side is limited to what was annotated as sand, rubble, background or marine species.

In [ ]:
# ---- what did the model call each annotated region? (raw model classes, pooled over frames) ----
from collections import Counter

CORAL_CLASSES = ["branching", "massive", "tabular", "encrusting", "foliose", "corymbose"]
coral_idx = [names.index(n) for n in CORAL_CLASSES if n in names]
coral_raw = np.array([i for i, n in sorted(id2label.items())
                      if (lut[i] != 255 and lut[i] in coral_idx) or "other coral" in n or "hard coral" in n])
print("model classes counted as 'coral':", [id2label[i] for i in coral_raw])

per_class = {k: Counter() for k in range(K)}
tp = fn = fp = tn = 0
for s in kept:
    ids = RAW_IDS[s]
    gt = np.array(Image.open(own / "masks" / f"{s}.png"))
    valid = gt != IGN
    for k in np.unique(gt[valid]):
        v, n = np.unique(ids[gt == k], return_counts=True)
        per_class[int(k)].update(dict(zip(v.tolist(), n.tolist())))
    g = np.isin(gt, coral_idx)[valid]
    p = np.isin(ids, coral_raw)[valid]
    tp += int((g & p).sum()); fn += int((g & ~p).sum()); fp += int((~g & p).sum()); tn += int((~g & ~p).sum())

rows = []
for k in range(K):
    c = per_class[k]; n = sum(c.values())
    if n == 0:
        continue
    called_coral = sum(v for i, v in c.items() if i in set(coral_raw.tolist())) / n
    rows.append({"annotated class": names[k], "annotated px": n,
                 "called any coral": f"{called_coral:.0%}" if names[k] in CORAL_CLASSES else "-",
                 "model's top classes on these pixels": ", ".join(
                     f"{id2label[i]} {v / n:.0%}" for i, v in c.most_common(5))})
pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(rows).set_index("annotated class"))

print(f"\nCoral vs not-coral on the annotated pixels (all {len(kept)} frames):")
print(f"  coral recall    {tp / max(tp + fn, 1):.1%}   (of pixels annotated as coral, how many the model called some coral class)")
print(f"  coral precision {tp / max(tp + fp, 1):.1%}   (optimistic: only counts non-coral that was annotated; see note above)")
print(f"  not-coral recall {tn / max(tn + fp, 1):.1%}  (of the sand/rubble/background/marine pixels, how many the model did NOT call coral)")